# TP02: Cleaning Data - Applying OOP and Design Patterns

Dataset: `sample_data_2.csv` (7 rows, columns: id, name, age, height_cm, weight_kg, city, score).
Each exercise builds on the previous one, ending with a complete cleaning pipeline.

In [18]:
import time
import functools
from abc import ABC, abstractmethod

import pandas as pd

DATA_PATH = "/content/sample_data (2).csv"

## Exercise 1: CSVReader Class

The class encapsulates everything related to reading a CSV file. The file path and the loaded
DataFrame are stored as attributes; `read()` loads the data and `preview(n)` prints the first n rows.

In [19]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self._df = None

    def read(self):
        self._df = pd.read_csv(self.file_path)
        return self._df

    def preview(self, n=5):
        if self._df is None:
            self.read()
        print(self._df.head(n))

    @property
    def data(self):
        return self._df


reader = CSVReader(DATA_PATH)
df = reader.read()
reader.preview(7)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles    NaN
6   7    Grace   NaN      162.0       60.0      Chicago   92.0


In [20]:
print("Shape:", df.shape)
print()
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())

Shape: (7, 7)

id             int64
name          object
age          float64
height_cm    float64
weight_kg    float64
city          object
score        float64
dtype: object

Missing values per column:
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64


**Interpretation:** The file has 7 rows and 7 columns. Four numeric columns contain missing values:
age (3), height_cm (1), weight_kg (1) and score (2). Only 2 of the 7 rows are complete, so the
choice of missing-value strategy will have a large effect on the result.

## Exercise 2: Strategy Pattern for Missing Values

`MissingValueStrategy` defines a common interface `handle(df)`. Each subclass implements a different
behaviour, and `DataCleaner` applies whichever strategy it is given, so the strategy can be switched
without changing the cleaner.

In [21]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    def handle(self, df):
        df = df.copy()
        num_cols = df.select_dtypes(include="number").columns
        df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
        return df


class FillMode(MissingValueStrategy):
    def handle(self, df):
        df = df.copy()
        for col in df.columns:
            mode = df[col].mode()
            if not mode.empty:
                df[col] = df[col].fillna(mode.iloc[0])
        return df


class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

In [22]:
cleaner = DataCleaner(DropMissing())
print("DropMissing")
print(cleaner.clean(df))

DropMissing
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
1   5    Eva  27.0      160.0       55.0  New York   88.0


In [23]:
cleaner.set_strategy(FillMean())
print("FillMean")
print(cleaner.clean(df))
print()
print("Column means used:")
print(df[["age", "height_cm", "weight_kg", "score"]].mean())

FillMean
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0

Column means used:
age           30.25
height_cm    167.00
weight_kg     70.00
score         86.00
dtype: float64


In [24]:
cleaner.set_strategy(FillMode())
print("FillMode")
print(cleaner.clean(df))

FillMode
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles   75.0
6   7    Grace  27.0      162.0       60.0      Chicago   92.0


**Interpretation:**

- **DropMissing** keeps only Alice and Eva (2 of 7 rows), losing about 71% of the data. Not suitable here.
- **FillMean** keeps all 7 rows and fills each gap with the column average (age 30.25, height 167.0,
  weight 70.0, score 86.0). This is the most reasonable option for this small numeric dataset.
- **FillMode** also keeps all rows, but no numeric value repeats in this data, so the "mode" is just the
  smallest value (e.g. age 27, score 75). Mode filling makes more sense for categorical columns.

The same `DataCleaner` object produced all three results; only the strategy changed.

## Exercise 3: Decorators for Logging and Timing

`@log_action` prints when a method starts and finishes; `@log_time` prints its execution time.
They are applied to subclasses of `CSVReader` and `DataCleaner`, so the original classes stay
unchanged (Open/Closed Principle).

In [25]:
def log_action(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Starting {func.__qualname__}")
        result = func(*args, **kwargs)
        print(f"[LOG] Finished {func.__qualname__}")
        return result
    return wrapper


def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter() - start) * 1000
        print(f"[TIME] {func.__qualname__} took {elapsed:.3f} ms")
        return result
    return wrapper


class LoggedCSVReader(CSVReader):
    @log_action
    @log_time
    def read(self):
        return super().read()


class LoggedDataCleaner(DataCleaner):
    @log_action
    @log_time
    def clean(self, df):
        return super().clean(df)

In [26]:
logged_reader = LoggedCSVReader(DATA_PATH)
raw = logged_reader.read()
print()
logged_cleaner = LoggedDataCleaner(FillMean())
cleaned = logged_cleaner.clean(raw)
print()
print("Remaining missing values:", int(cleaned.isna().sum().sum()))

[LOG] Starting LoggedCSVReader.read
[TIME] LoggedCSVReader.read took 0.802 ms
[LOG] Finished LoggedCSVReader.read

[LOG] Starting LoggedDataCleaner.clean
[TIME] LoggedDataCleaner.clean took 1.022 ms
[LOG] Finished LoggedDataCleaner.clean

Remaining missing values: 0


**Interpretation:** The log messages and timings were added without modifying `CSVReader` or
`DataCleaner`. Both operations run in a few milliseconds because the file is tiny. After FillMean,
no missing values remain.

## Exercise 4: Factory Pattern for Data Transformations

`DataTransform` defines `apply(df)`. Three transformations are implemented, and `TransformFactory`
creates the right object from a string name, so the calling code never instantiates the classes directly.

- `NormalizeColumns`: min-max scales numeric columns to [0, 1] (the `id` column is excluded).
- `RemoveDuplicates`: drops duplicate rows.
- `StandardizeText`: trims whitespace and converts text columns to lowercase.

In [27]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass


class NormalizeColumns(DataTransform):
    def __init__(self, exclude=("id",)):
        self.exclude = list(exclude)

    def apply(self, df):
        df = df.copy()
        cols = [c for c in df.select_dtypes(include="number").columns if c not in self.exclude]
        for c in cols:
            min_v, max_v = df[c].min(), df[c].max()
            df[c] = 0.0 if max_v == min_v else (df[c] - min_v) / (max_v - min_v)
        return df


class RemoveDuplicates(DataTransform):
    def apply(self, df):
        before = len(df)
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"RemoveDuplicates: {before - len(df)} duplicate row(s) removed")
        return df


class StandardizeText(DataTransform):
    def apply(self, df):
        df = df.copy()
        for c in [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]:
            df[c] = df[c].str.strip().str.lower()
        return df


class TransformFactory:
    _registry = {
        "normalize": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, name, **kwargs):
        key = name.lower()
        if key not in cls._registry:
            raise ValueError(f"Unknown transform '{name}'. Options: {list(cls._registry)}")
        return cls._registry[key](**kwargs)

In [28]:
transformed = cleaned
for name in ["remove_duplicates", "standardize_text", "normalize"]:
    transformed = TransformFactory.create(name).apply(transformed)

print(transformed.round(3))

RemoveDuplicates: 0 duplicate row(s) removed
   id     name    age  height_cm  weight_kg         city  score
0   1    alice  0.250      0.333      0.433     new york  0.588
1   2      bob  0.406      0.800      0.500  los angeles  0.882
2   3  charlie  1.000      0.533      0.567      chicago  0.647
3   4    david  0.406      0.467      0.833      houston  0.000
4   5      eva  0.000      0.000      0.000     new york  0.765
5   6    frank  0.375      1.000      1.000  los angeles  0.647
6   7    grace  0.406      0.133      0.167      chicago  1.000


In [29]:
try:
    TransformFactory.create("unknown")
except ValueError as e:
    print("Error:", e)

Error: Unknown transform 'unknown'. Options: ['normalize', 'remove_duplicates', 'standardize_text']


**Interpretation:**

- No duplicate rows exist in this dataset, so `RemoveDuplicates` removed 0 rows.
- Names and cities are now lowercase, which avoids mismatches such as "New York" vs "new york".
- After normalisation every numeric column lies between 0 and 1. For example, Frank has the highest
  height and weight (1.0), and Eva has the lowest (0.0). Mean-filled values sit in the middle
  (e.g. imputed weight 70 becomes 0.5).
- Requesting an unknown name raises a clear error instead of failing silently.

## Exercise 5: Full Cleaning Pipeline (Template Method Pattern)

`DataPipeline.run()` fixes the order of the steps: load, clean, transform, save. `CSVDataPipeline`
implements each step using the classes from the previous exercises.

In [30]:
class DataPipeline(ABC):
    def run(self):
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.df

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, strategy, transforms):
        self.reader = LoggedCSVReader(input_path)
        self.cleaner = LoggedDataCleaner(strategy)
        self.transforms = transforms
        self.output_path = output_path
        self.df = None

    def load(self):
        self.df = self.reader.read()

    def clean(self):
        self.df = self.cleaner.clean(self.df)

    @log_action
    def transform(self):
        for name in self.transforms:
            self.df = TransformFactory.create(name).apply(self.df)

    @log_action
    def save(self):
        self.df.to_csv(self.output_path, index=False)
        print(f"Saved {self.df.shape[0]} rows x {self.df.shape[1]} columns to {self.output_path}")

In [31]:
pipeline = CSVDataPipeline(
    input_path=DATA_PATH,
    output_path="cleaned_data.csv",
    strategy=FillMean(),
    transforms=["remove_duplicates", "standardize_text", "normalize"],
)
result = pipeline.run()

[LOG] Starting LoggedCSVReader.read
[TIME] LoggedCSVReader.read took 0.672 ms
[LOG] Finished LoggedCSVReader.read
[LOG] Starting LoggedDataCleaner.clean
[TIME] LoggedDataCleaner.clean took 0.998 ms
[LOG] Finished LoggedDataCleaner.clean
[LOG] Starting CSVDataPipeline.transform
RemoveDuplicates: 0 duplicate row(s) removed
[LOG] Finished CSVDataPipeline.transform
[LOG] Starting CSVDataPipeline.save
Saved 7 rows x 7 columns to cleaned_data.csv
[LOG] Finished CSVDataPipeline.save


In [32]:
print(pd.read_csv("cleaned_data.csv").round(3))
print()
print("Missing values in saved file:", int(pd.read_csv("cleaned_data.csv").isna().sum().sum()))

   id     name    age  height_cm  weight_kg         city  score
0   1    alice  0.250      0.333      0.433     new york  0.588
1   2      bob  0.406      0.800      0.500  los angeles  0.882
2   3  charlie  1.000      0.533      0.567      chicago  0.647
3   4    david  0.406      0.467      0.833      houston  0.000
4   5      eva  0.000      0.000      0.000     new york  0.765
5   6    frank  0.375      1.000      1.000  los angeles  0.647
6   7    grace  0.406      0.133      0.167      chicago  1.000

Missing values in saved file: 0


**Interpretation:** The pipeline ran all four steps in a fixed order and saved `cleaned_data.csv`
with all 7 rows, no missing values, lowercase text and numeric features scaled to [0, 1]. The output
matches the step-by-step result from Exercise 4, confirming the pipeline combines the earlier
components correctly. A different strategy or list of transforms can be passed in without changing
the pipeline code.